# INGESTION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def load_raw_data(csv_path):
    """
    Load raw CSV data.
    
    Args:
        csv_path (str): Path to CSV file
        
    Returns:
        pd.DataFrame: Raw dataframe
    """
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded {len(df)} rows from {csv_path}")
    return df

# Example usage
df_raw = load_raw_data("20250815-sensor-data.csv")
df_raw.head()

# PREPROCESSING

In [ ]:
def prepare_timestamp(df, utc_col='Timestamp', tz='Asia/Ho_Chi_Minh'):
    """
    Convert UTC timestamp to local timezone.
    
    Args:
        df (pd.DataFrame): Input dataframe
        utc_col (str): Name of UTC timestamp column
        tz (str): Target timezone
        
    Returns:
        pd.DataFrame: Dataframe with Timestamp_vn column
    """
    df = df.copy()
    df[utc_col] = pd.to_datetime(df[utc_col], errors='coerce', utc=True)
    df = df.sort_values(utc_col).reset_index(drop=True)
    
    # Convert to local timezone
    df['Timestamp_local'] = df[utc_col].dt.tz_convert(tz)
    df['Timestamp_vn'] = df['Timestamp_local'].dt.tz_localize(None)
    
    print(f"✓ Converted timestamps to {tz}")
    print(f"  Time range: {df['Timestamp_vn'].min()} to {df['Timestamp_vn'].max()}")
    return df

# Apply
df_time = prepare_timestamp(df_raw)

In [ ]:
def compute_norms(df, accel_cols=('Accel_x','Accel_y','Accel_z'),
                       gyro_cols=('Gyro_x','Gyro_y','Gyro_z')):
    """
    Compute Euclidean norms for accelerometer and gyroscope.
    
    Args:
        df (pd.DataFrame): Input dataframe
        accel_cols (tuple): Accelerometer column names
        gyro_cols (tuple): Gyroscope column names
        
    Returns:
        pd.DataFrame: Dataframe with acc_norm and gyro_norm columns
    """
    df = df.copy()
    acc_values = df[list(accel_cols)].values
    gyro_values = df[list(gyro_cols)].values
    
    df['acc_norm'] = np.linalg.norm(acc_values, axis=1)
    df['gyro_norm'] = np.linalg.norm(gyro_values, axis=1)
    
    print(f"✓ Computed norms")
    print(f"  acc_norm range: [{df['acc_norm'].min():.2f}, {df['acc_norm'].max():.2f}]")
    print(f"  gyro_norm range: [{df['gyro_norm'].min():.2f}, {df['gyro_norm'].max():.2f}]")
    return df

# Apply
df_norms = compute_norms(df_time)

# FEATURE ENGINEERING

In [ ]:
def compute_activity_score(df, weights=(0.7, 0.3), 
                           on_pct=75, off_pct=60, eps=1e-6):
    """
    Compute activity score using robust normalization.
    Returns dataframe with score + learned thresholds.
    
    Args:
        df (pd.DataFrame): Dataframe with acc_norm, gyro_norm
        weights (tuple): (weight_acc, weight_gyro)
        on_pct (float): Percentile for ON threshold
        off_pct (float): Percentile for OFF threshold
        eps (float): Small value to avoid division by zero
        
    Returns:
        tuple: (df_with_score, params_dict)
    """
    df = df.copy()
    
    # Compute robust statistics (median and MAD)
    acc_med = np.median(df['acc_norm'].values)
    gyro_med = np.median(df['gyro_norm'].values)
    
    acc_mad = np.median(np.abs(df['acc_norm'].values - acc_med)) + eps
    gyro_mad = np.median(np.abs(df['gyro_norm'].values - gyro_med)) + eps
    
    # Robust deviation scores
    z_acc = np.abs((df['acc_norm'].values - acc_med) / acc_mad)
    z_gyro = np.abs((df['gyro_norm'].values - gyro_med) / gyro_mad)
    
    # Weighted activity score
    w_acc, w_gyro = weights
    activity_score = w_acc * z_acc + w_gyro * z_gyro
    df['activity_score'] = activity_score
    
    # Compute thresholds from percentiles
    on_th = np.percentile(activity_score, on_pct)
    off_th = np.percentile(activity_score, off_pct)
    
    params = {
        'acc_med': acc_med,
        'acc_mad': acc_mad,
        'gyro_med': gyro_med,
        'gyro_mad': gyro_mad,
        'weights': weights,
        'on_th': on_th,
        'off_th': off_th,
        'on_pct': on_pct,
        'off_pct': off_pct
    }
    
    print(f"✓ Computed activity score")
    print(f"  Thresholds: ON={on_th:.2f} (P{on_pct}), OFF={off_th:.2f} (P{off_pct})")
    
    return df, params

# Apply - Training phase
df_scored, train_params = compute_activity_score(df_clean, 
                                                  weights=(0.7, 0.3),
                                                  on_pct=75, 
                                                  off_pct=60)

# Save parameters for inference on new data
import json
with open('activity_params.json', 'w') as f:
    json.dump(train_params, f, indent=2)
print("✓ Saved training parameters")

# CLASSIFICATION

In [ ]:
def apply_hysteresis(df, on_th, off_th, debounce_sec=10.0, fs=1.0):
    """
    Apply hysteresis with debounce to activity score.
    
    Args:
        df (pd.DataFrame): Dataframe with activity_score column
        on_th (float): Threshold to turn ON
        off_th (float): Threshold to turn OFF
        debounce_sec (float): Seconds to wait before turning OFF
        fs (float): Sampling frequency (Hz)
        
    Returns:
        pd.DataFrame: Dataframe with active_cont column (0/1)
    """
    df = df.copy()
    
    debounce_samples = int(round(debounce_sec * fs))
    
    states = []
    active = False
    below_count = 0
    
    for score in df['activity_score'].values:
        if not active:
            # Turn ON if score exceeds ON threshold
            if score > on_th:
                active = True
                below_count = 0
        else:
            # Count consecutive samples below OFF threshold
            if score < off_th:
                below_count += 1
                # Turn OFF only after debounce period
                if below_count >= debounce_samples:
                    active = False
                    below_count = 0
            else:
                below_count = 0
        
        states.append(1 if active else 0)
    
    df['active_cont'] = states
    
    active_pct = np.mean(states) * 100
    print(f"✓ Applied hysteresis (debounce={debounce_sec}s)")
    print(f"  Active: {active_pct:.1f}%, Idle: {100-active_pct:.1f}%")
    
    return df

# Apply - Inference phase
df_classified = apply_hysteresis(df_scored, 
                                 on_th=train_params['on_th'],
                                 off_th=train_params['off_th'],
                                 debounce_sec=10.0)

# VISUALIZATION 

In [ ]:
def plot_results(df, on_th=None, off_th=None):
    """
    Plot activity score and classification results.
    
    Args:
        df (pd.DataFrame): Dataframe with activity_score and active_cont
        on_th (float): ON threshold (optional, for visualization)
        off_th (float): OFF threshold (optional, for visualization)
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Plot 1: Activity Score
    axes[0].plot(df['Timestamp_vn'], df['activity_score'], 
                 color='navy', linewidth=1.2, label='Activity Score')
    if on_th is not None:
        axes[0].axhline(on_th, color='green', linestyle='--', 
                       linewidth=1, label=f'ON threshold = {on_th:.2f}')
    if off_th is not None:
        axes[0].axhline(off_th, color='red', linestyle='--', 
                       linewidth=1, label=f'OFF threshold = {off_th:.2f}')
    axes[0].set_ylabel('Activity Score')
    axes[0].set_title('Activity Score với Hysteresis Thresholds')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Classification
    axes[1].fill_between(df['Timestamp_vn'], df['active_cont'], 
                         step='post', alpha=0.5, color='green',
                         label='Active (1) / Idle (0)')
    axes[1].set_ylabel('State')
    axes[1].set_ylim(-0.2, 1.2)
    axes[1].set_title('Kết quả Phân loại Active/Idle')
    axes[1].legend(loc='upper left')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Norms
    axes[2].plot(df['Timestamp_vn'], df['acc_norm'], 
                 color='black', linewidth=1, alpha=0.7, label='acc_norm')
    axes[2].plot(df['Timestamp_vn'], df['gyro_norm'], 
                 color='purple', linewidth=1, alpha=0.7, label='gyro_norm')
    axes[2].set_ylabel('Magnitude')
    axes[2].set_xlabel('Time (VN)')
    axes[2].set_title('IMU Norms (Accel & Gyro)')
    axes[2].legend(loc='upper left')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print("✓ Generated visualization")

# Apply
plot_results(df_classified, 
             on_th=train_params['on_th'],
             off_th=train_params['off_th'])

In [ ]:
# Save classified data
output_path = "20250803_classified.csv"
df_
classified.to_csv(output_path, index=False)
print(f"✓ Saved classified data to {output_path}")

# Save parameters for future inference
params_path = "../data/processed/activity_params.json"
with open(params_path, 'w') as f:
    json.dump(train_params, f, indent=2)
print(f"✓ Saved parameters to {params_path}")